# 04 — Performance & Evaluation

Two critical production concerns: making your LLM calls faster/cheaper with caching, and programmatically evaluating the quality of your RAG pipeline.

## Setup

In [ ]:
import os
import time
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embed_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

print("Models ready")

## Step 17 — LLM Caching

Every identical LLM call wastes tokens and time. LangChain's `InMemoryCache` intercepts duplicate queries and returns cached results instantly.

In [ ]:
from langchain.globals import set_llm_cache
from langchain_community.cache import InMemoryCache

set_llm_cache(InMemoryCache())
print("Cache enabled")

In [ ]:
question = "What is the capital of Indonesia?"

# First call — hits the API
start = time.perf_counter()
result1 = llm.invoke(question)
t1 = time.perf_counter() - start
print(f"First call:  {t1:.3f}s")
print(result1.content[:100] + "...")

# Second call — served from cache
start = time.perf_counter()
result2 = llm.invoke(question)
t2 = time.perf_counter() - start
print(f"\nSecond call: {t2:.3f}s")
print(result2.content[:100] + "...")

print(f"\nSpeedup: {t1 / max(t2, 0.001):.0f}x faster!")

## Step 18b — Automated RAG Evaluation (LLM-as-a-Judge)

Manual evaluation doesn't scale. We use **DeepEval** with Gemini as the judge to programmatically score our RAG pipeline on the **RAG Triad**:

1. **Faithfulness** — Does the answer contradict the retrieved context?
2. **Context Relevance** — Was the retrieved context useful for answering the query?

In [ ]:
# Build a minimal RAG pipeline for evaluation
from langchain.chains import RetrievalQA
from pathlib import Path

doc_dir = Path("../docs/dummy-knowledge")
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

all_texts, all_metas = [], []
for fp in sorted(doc_dir.glob("*.md")):
    content = fp.read_text()
    for c in splitter.split_text(content):
        all_texts.append(c)
        all_metas.append({"source": fp.name})

db = QdrantVectorStore.from_texts(
    all_texts, embed_model, metadatas=all_metas, location=":memory:"
)
rag = RetrievalQA.from_chain_type(
    llm=llm, retriever=db.as_retriever(search_kwargs={"k": 2})
)

print("RAG pipeline ready for evaluation")

### Faithfulness Evaluation

We ask a question, capture the answer and retrieved context, then have Gemini-as-a-judge score whether the answer is faithful to the context.

In [ ]:
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# Run a RAG query
query = "What is the rate limit for the Quantum Weather API free tier?"
result = rag.invoke({"query": query})
answer = result["result"]
retriever = db.as_retriever(search_kwargs={"k": 2})
context_docs = retriever.invoke(query)
context_texts = [d.page_content for d in context_docs]

# Create the test case
test_case = LLMTestCase(
    input=query,
    actual_output=answer,
    retrieval_context=context_texts,
)

# Evaluate with Gemini as judge
metric = FaithfulnessMetric(
    threshold=0.7,
    model="gemini/gemini-2.5-flash",
)

metric.measure(test_case)
print(f"Query: {query}")
print(f"\nAnswer: {answer[:200]}...")
print(f"\nFaithfulness score: {metric.score:.2f} (threshold: 0.7)")
print(f"Reason: {metric.reason}")

### Context Relevance Evaluation

Now we check whether the retrieved context was actually relevant to answering the question.

In [ ]:
from deepeval.metrics import ContextRelevanceMetric

relevance_metric = ContextRelevanceMetric(
    threshold=0.7,
    model="gemini/gemini-2.5-flash",
)

relevance_metric.measure(test_case)
print(f"Query: {query}")
print(f"\nContext Relevance score: {relevance_metric.score:.2f} (threshold: 0.7)")
print(f"Reason: {relevance_metric.reason}")

### Bonus: Test with an Out-of-Context Query

Ask about something NOT in our documents — the faithfulness score should drop.

In [ ]:
bad_query = "What is the maintenance schedule for the Quantum Weather API servers?"
bad_result = rag.invoke({"query": bad_query})
bad_answer = bad_result["result"]
bad_docs = retriever.invoke(bad_query)
bad_context = [d.page_content for d in bad_docs]

bad_case = LLMTestCase(
    input=bad_query,
    actual_output=bad_answer,
    retrieval_context=bad_context,
)

bad_metric = FaithfulnessMetric(threshold=0.7, model="gemini/gemini-2.5-flash")
bad_metric.measure(bad_case)

print(f"Query: {bad_query}")
print(f"\nAnswer: {bad_answer[:200]}...")
print(f"\nFaithfulness score: {bad_metric.score:.2f}")
print(f"Reason: {bad_metric.reason}")

### Summary

| Metric | High score means | Purpose |
|--------|------------------|---------|
| Faithfulness | Answer matches retrieved context | Prevents hallucination |
| Context Relevance | Retrieved docs are on-topic | Ensures good retrieval |

In production, you'd run these metrics across a test suite of 50-100 queries and track scores over time to catch regressions.